# Using Machine Learning techniques to predict a stroke

## Imports

In [341]:
import kagglehub
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report
from sklearn.base import clone
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

## Load in data, basic data information

In [342]:
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")

file_path = os.path.join(path, "healthcare-dataset-stroke-data.csv")
df = pd.read_csv(file_path)

df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [343]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   str    
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   str    
 6   work_type          5110 non-null   str    
 7   Residence_type     5110 non-null   str    
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   str    
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), str(5)
memory usage: 479.2 KB


## Data cleaning

In [344]:
df.columns

Index(['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'Residence_type', 'avg_glucose_level', 'bmi',
       'smoking_status', 'stroke'],
      dtype='str')

### Checking data for unique values

Before modifying the data to be fit for training we should check how many unique values each feature can have, so that we can determine how to represent it (eg. use 0 and 1 for yes-no columns or one-hot encoding for those with more posiible values).

In [345]:
for column in df.columns:
    print(f"{column} values: {df[column].unique()}")

id values: [ 9046 51676 31112 ... 19723 37544 44679]
gender values: <StringArray>
['Male', 'Female', 'Other']
Length: 3, dtype: str
age values: [6.70e+01 6.10e+01 8.00e+01 4.90e+01 7.90e+01 8.10e+01 7.40e+01 6.90e+01
 5.90e+01 7.80e+01 5.40e+01 5.00e+01 6.40e+01 7.50e+01 6.00e+01 5.70e+01
 7.10e+01 5.20e+01 8.20e+01 6.50e+01 5.80e+01 4.20e+01 4.80e+01 7.20e+01
 6.30e+01 7.60e+01 3.90e+01 7.70e+01 7.30e+01 5.60e+01 4.50e+01 7.00e+01
 6.60e+01 5.10e+01 4.30e+01 6.80e+01 4.70e+01 5.30e+01 3.80e+01 5.50e+01
 1.32e+00 4.60e+01 3.20e+01 1.40e+01 3.00e+00 8.00e+00 3.70e+01 4.00e+01
 3.50e+01 2.00e+01 4.40e+01 2.50e+01 2.70e+01 2.30e+01 1.70e+01 1.30e+01
 4.00e+00 1.60e+01 2.20e+01 3.00e+01 2.90e+01 1.10e+01 2.10e+01 1.80e+01
 3.30e+01 2.40e+01 3.40e+01 3.60e+01 6.40e-01 4.10e+01 8.80e-01 5.00e+00
 2.60e+01 3.10e+01 7.00e+00 1.20e+01 6.20e+01 2.00e+00 9.00e+00 1.50e+01
 2.80e+01 1.00e+01 1.80e+00 3.20e-01 1.08e+00 1.90e+01 6.00e+00 1.16e+00
 1.00e+00 1.40e+00 1.72e+00 2.40e-01 1.64e+00 1.56e+0

### Adjusting column types

In [346]:
df["age"] = df["age"].astype("uint8")
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80,1,0,Yes,Private,Urban,83.75,NaN,never smoked,0
5106,44873,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.0,never smoked,0
5107,19723,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.6,never smoked,0
5108,37544,Male,51,0,0,Yes,Private,Rural,166.29,25.6,formerly smoked,0


### Checking for NaN values

We check if there are any values missing from the dataset.
- `df.isna()` – returns a DataFrame of True/False values (True if the value in the original DataFrame is NaN, False otherwise)  
- `df.isna().any()` – for each column, checks if it contains at least one True value  
- `df.isna().any().any()` – returns True if there is **at least one NaN** in the original DataFrame, False otherwise

In [347]:
df.isna().any().any()

np.True_

### Dealing with NaNs

BMI is the only columns with missing values. We fill them using the gender average.

In [348]:
df["bmi"] = df["bmi"].fillna(df.groupby("gender")["bmi"].transform("mean"))
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67,0,1,Yes,Private,Urban,228.69,36.600000,formerly smoked,1
1,51676,Female,61,0,0,Yes,Self-employed,Rural,202.21,29.065758,never smoked,1
2,31112,Male,80,0,1,Yes,Private,Rural,105.92,32.500000,never smoked,1
3,60182,Female,49,0,0,Yes,Private,Urban,171.23,34.400000,smokes,1
4,1665,Female,79,1,0,Yes,Self-employed,Rural,174.12,24.000000,never smoked,1
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,18234,Female,80,1,0,Yes,Private,Urban,83.75,29.065758,never smoked,0
5106,44873,Female,81,0,0,Yes,Self-employed,Urban,125.20,40.000000,never smoked,0
5107,19723,Female,35,0,0,Yes,Self-employed,Rural,82.99,30.600000,never smoked,0
5108,37544,Male,51,0,0,Yes,Private,Rural,166.29,25.600000,formerly smoked,0


### Modifying the data

In [349]:
# Binary to 1/0
df["ever_married"] = df["ever_married"].map({"Yes": 1, "No": 0})

# One-Hot Encoding
df = pd.get_dummies(df, columns=["gender", "work_type", "Residence_type", "smoking_status"])

# Convert all bool columns from OHE to int
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,9046,67,0,1,1,228.69,36.600000,1,0,1,...,0,1,0,0,0,1,0,1,0,0
1,51676,61,0,0,1,202.21,29.065758,1,1,0,...,0,0,1,0,1,0,0,0,1,0
2,31112,80,0,1,1,105.92,32.500000,1,0,1,...,0,1,0,0,1,0,0,0,1,0
3,60182,49,0,0,1,171.23,34.400000,1,1,0,...,0,1,0,0,0,1,0,0,0,1
4,1665,79,1,0,1,174.12,24.000000,1,1,0,...,0,0,1,0,1,0,0,0,1,0


In [350]:
# Move label to the rightmost side of the dataframe
cols = [c for c in df.columns if c != "stroke"] + ["stroke"]
df = df[cols]

### Renaming the columns

In this step we standardize the naming convention of features in the dataframe.

In [351]:
def get_renamed_column(name):
    name = name.strip()
    name = name.lower()
    name = name.replace(" ", "_")

    return name


def get_rename_dict(column_names):
    rename_dict = {}
    for name in column_names:
        rename_dict[name] = get_renamed_column(name)

    return rename_dict

In [352]:
df = df.rename(columns=get_rename_dict(df.columns))
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,gender_female,gender_male,gender_other,...,work_type_private,work_type_self-employed,work_type_children,residence_type_rural,residence_type_urban,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,stroke
0,9046,67,0,1,1,228.69,36.600000,0,1,0,...,1,0,0,0,1,0,1,0,0,1
1,51676,61,0,0,1,202.21,29.065758,1,0,0,...,0,1,0,1,0,0,0,1,0,1
2,31112,80,0,1,1,105.92,32.500000,0,1,0,...,1,0,0,1,0,0,0,1,0,1
3,60182,49,0,0,1,171.23,34.400000,1,0,0,...,1,0,0,0,1,0,0,0,1,1
4,1665,79,1,0,1,174.12,24.000000,1,0,0,...,0,1,0,1,0,0,0,1,0,1


### Check for duplicates

In [353]:
any(df.duplicated())

False

There are only unique records in the dataset already, we may proceed.

### Check for class imbalance

Ideally here should be a similar number of records in each class.

In [354]:
df[df.columns[-1]].sum() / df[df.columns[-1]].size

np.float64(0.0487279843444227)

Since stroke was confirmed only in 5% of patients we will use a few different techniques deal with the imbalance.

## Preparing the train and test sets

### Train-test split

In [355]:
print(df.columns)

Index(['id', 'age', 'hypertension', 'heart_disease', 'ever_married',
       'avg_glucose_level', 'bmi', 'gender_female', 'gender_male',
       'gender_other', 'work_type_govt_job', 'work_type_never_worked',
       'work_type_private', 'work_type_self-employed', 'work_type_children',
       'residence_type_rural', 'residence_type_urban',
       'smoking_status_unknown', 'smoking_status_formerly_smoked',
       'smoking_status_never_smoked', 'smoking_status_smokes', 'stroke'],
      dtype='str')


In [356]:
columns_to_drop = ["ever_married", "gender_female", "gender_male", "gender_other", "work_type_govt_job", "work_type_never_worked",
       "work_type_private", "work_type_self-employed", "work_type_children"]
df = df.drop(columns=columns_to_drop)

In [357]:
X = df.drop(columns=[df.columns[-1]]) 
y = df[df.columns[-1]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Create models

### Random Forests

In [358]:
random_forest_models = {
    "RandomForest_100_5": RandomForestClassifier(
        n_estimators=100,
        max_depth=5
    ),
    "RandomForest_50_5": RandomForestClassifier(
        n_estimators=50,
        max_depth=5
    ),
    "RandomForest_50_10": RandomForestClassifier(
        n_estimators=50,
        max_depth=10
    ),
    "RandomForest_500_3": RandomForestClassifier(
        n_estimators=500,
        max_depth=3
    ),
    "RandomForest_200_10": RandomForestClassifier(
        n_estimators=200,
        max_depth=10
    ),
    "RandomForest_300_7": RandomForestClassifier(
        n_estimators=300,
        max_depth=7
    ),
}

### Neural Networks

In [359]:
input_shape = X_train.shape[1]

neural_networks_models = {
    "NN_64_32_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_32_32_32_1": Sequential([
        Dense(32, activation="sigmoid", input_shape=(input_shape,)),
        Dense(32, activation="sigmoid"),
        Dense(32, activation="sigmoid"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(64, activation="tanh"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_128_64_64_1": Sequential([
        Dense(128, activation="relu", input_shape=(input_shape,)),
        Dense(64, activation="relu"),
        Dense(64, activation="relu"),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(1, activation="sigmoid")
    ]),
    "NN_64_32_64_1": Sequential([
        Dense(64, activation="relu", input_shape=(input_shape,)),
        Dense(32, activation="tanh"),
        Dense(64, activation="tanh"),
        Dense(1, activation="sigmoid")
    ]),
}

c:\Users\Dell\anaconda\envs\tensorflow\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Train models

### Using SMOTE to deal with class imbalance

#### Cross-validation pipeline

In [360]:
def validate_models_smote(models, X, y):
    results = {}

    for name, model in models.items():
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("smote", SMOTE(sampling_strategy="minority")),
            ("model", model)
        ])

        scores = cross_validate(pipeline, X, y, cv=5, scoring=["precision", "recall", "f1"])
        results[name] = {
            "precision": scores["test_precision"].mean(),
            "recall": scores["test_recall"].mean(),
            "f1": scores["test_f1"].mean(),
        }

    return results

In [361]:
results = validate_models_smote(random_forest_models, X_train, y_train)
for name, result in results.items():
    print(f"{name}: {result}\n")

RandomForest_100_5: {'precision': np.float64(0.11538325995754026), 'recall': np.float64(0.759317211948791), 'f1': np.float64(0.200275389176938)}

RandomForest_50_5: {'precision': np.float64(0.11811092355340817), 'recall': np.float64(0.7701280227596017), 'f1': np.float64(0.20470997084023818)}

RandomForest_50_10: {'precision': np.float64(0.11809729236564592), 'recall': np.float64(0.48036984352773826), 'f1': np.float64(0.1893341329390947)}

RandomForest_500_3: {'precision': np.float64(0.11102230909924637), 'recall': np.float64(0.8130867709815078), 'f1': np.float64(0.19531276386306534)}

RandomForest_200_10: {'precision': np.float64(0.115927203357308), 'recall': np.float64(0.4591749644381224), 'f1': np.float64(0.18462075770367764)}

RandomForest_300_7: {'precision': np.float64(0.12058561087639832), 'recall': np.float64(0.6681365576102418), 'f1': np.float64(0.2041737310889158)}



#### Training pipeline

In [362]:
def train_models_smote(models, X, y):

    trained_models = {}

    for name, model in models.items():
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("smote", SMOTE(sampling_strategy="minority")),
            ("model", model)
        ])

        trained_pipeline = pipeline.fit(X, y)
        trained_models[name] = trained_pipeline

    return trained_models


In [363]:
trained_models = train_models_smote(random_forest_models, X_train, y_train)

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"{name} Accuracy: {accuracy}")
    print(f"{name} Precision: {precision}")
    print(f"{name} Recall: {recall}")
    print(f"{name} F1: {f1}\n")

RandomForest_100_5 Accuracy: 0.7240704500978473
RandomForest_100_5 Precision: 0.15625
RandomForest_100_5 Recall: 0.8064516129032258
RandomForest_100_5 F1: 0.2617801047120419

RandomForest_50_5 Accuracy: 0.7309197651663405
RandomForest_50_5 Precision: 0.15309446254071662
RandomForest_50_5 Recall: 0.7580645161290323
RandomForest_50_5 F1: 0.25474254742547425

RandomForest_50_10 Accuracy: 0.7925636007827789
RandomForest_50_10 Precision: 0.12871287128712872
RandomForest_50_10 Recall: 0.41935483870967744
RandomForest_50_10 F1: 0.19696969696969696

RandomForest_500_3 Accuracy: 0.7064579256360078
RandomForest_500_3 Precision: 0.15
RandomForest_500_3 Recall: 0.8225806451612904
RandomForest_500_3 F1: 0.2537313432835821

RandomForest_200_10 Accuracy: 0.8023483365949119
RandomForest_200_10 Precision: 0.13541666666666666
RandomForest_200_10 Recall: 0.41935483870967744
RandomForest_200_10 F1: 0.2047244094488189

RandomForest_300_7 Accuracy: 0.7524461839530333
RandomForest_300_7 Precision: 0.15523465

### Using Ensemble to deal with class imbalance

In [364]:
ensemble_set = pd.concat((X_train.copy(), y_train.copy()), axis="columns")

# Divide the ensemble set by the label
ensemble_class_0 = ensemble_set[ensemble_set[ensemble_set.columns[-1]] == 0]
ensemble_class_1 = ensemble_set[ensemble_set[ensemble_set.columns[-1]] == 1]

# Create train and test sets
X_train_class_0 = ensemble_class_0.drop(ensemble_class_0.columns[-1], axis="columns")
y_train_class_0 = ensemble_class_0[ensemble_class_0.columns[-1]]

X_train_class_1 = ensemble_class_1.drop(ensemble_class_1.columns[-1], axis="columns")
y_train_class_1 = ensemble_class_1[ensemble_class_1.columns[-1]]

# Estimate the number of sub models required and the numer of class 0 records per each sub model
sub_models_count = int(len(X_train_class_0) / len(X_train_class_1))
records_per_set = len(X_train_class_0) // sub_models_count

ensemble_models = {}
for name, model in random_forest_models.items():
    ensemble_model = []
    for i in range(sub_models_count):
        X_chunk = X_train_class_0.sample(n=records_per_set, replace=True)
        y_chunk = y_train_class_0.loc[X_chunk.index]
        current_X_train = pd.concat((X_chunk, X_train_class_1), axis="rows")
        current_y_train = pd.concat((y_chunk, y_train_class_1), axis="rows")

        trained_model = clone(model)
        trained_model.fit(current_X_train, current_y_train)
        ensemble_model.append(trained_model)

    ensemble_models[name] = ensemble_model



In [365]:
for name, model in ensemble_models.items():
    votes = np.zeros(len(X_test))

    for sub_model in model:
        prob = sub_model.predict_proba(X_test)[:, 1]
        votes += prob

    avg_prob = votes / len(model)

    thresholds = np.linspace(0.01, 0.5, 50)
    for threshold in thresholds:
        print(f"CURRENT THRESHOLD: {threshold}")
        y_pred = (avg_prob > threshold).astype(int)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"{name} Accuracy: {accuracy}")
        print(f"{name} Precision: {precision}")
        print(f"{name} Recall: {recall}")
        print(f"{name} F1: {f1}\n")

CURRENT THRESHOLD: 0.01
RandomForest_100_5 Accuracy: 0.060665362035225046
RandomForest_100_5 Precision: 0.060665362035225046
RandomForest_100_5 Recall: 1.0
RandomForest_100_5 F1: 0.11439114391143912

CURRENT THRESHOLD: 0.02
RandomForest_100_5 Accuracy: 0.060665362035225046
RandomForest_100_5 Precision: 0.060665362035225046
RandomForest_100_5 Recall: 1.0
RandomForest_100_5 F1: 0.11439114391143912

CURRENT THRESHOLD: 0.03
RandomForest_100_5 Accuracy: 0.060665362035225046
RandomForest_100_5 Precision: 0.060665362035225046
RandomForest_100_5 Recall: 1.0
RandomForest_100_5 F1: 0.11439114391143912

CURRENT THRESHOLD: 0.04
RandomForest_100_5 Accuracy: 0.060665362035225046
RandomForest_100_5 Precision: 0.060665362035225046
RandomForest_100_5 Recall: 1.0
RandomForest_100_5 F1: 0.11439114391143912

CURRENT THRESHOLD: 0.05
RandomForest_100_5 Accuracy: 0.07045009784735812
RandomForest_100_5 Precision: 0.06126482213438735
RandomForest_100_5 Recall: 1.0
RandomForest_100_5 F1: 0.1154562383612663

CUR

### Using class weight to deal with the imbalance

#### Training Neural Networks

In [366]:
def train_neural_networks(models, X, y):
    results = {}

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    for name, model in models.items():
        model_copy = clone_model(model)
        model_copy.set_weights(model.get_weights()) 
        
        model_copy.compile(optimizer=Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["precision", "recall"])
        model_copy.fit(X_scaled, y, epochs=50, batch_size=32, verbose=0, class_weight={0: 1, 1: 19}, validation_split=0.2)

        results[name] = model_copy

    return results, scaler

In [367]:
trained_models, scaler = train_neural_networks(neural_networks_models, X_train, y_train)
X_test_scaled = scaler.transform(X_test)

for name, model in trained_models.items():
    y_prob = model.predict(X_test_scaled).flatten()
    thresholds = np.linspace(0.01, 0.9, 50)
    for threshold in thresholds:
        print(f"CURRENT THRESHOLD: {threshold}")
        y_pred = (y_prob > threshold).astype(int) 

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        print(f"{name} Accuracy: {accuracy}")
        print(f"{name} Precision: {precision}")
        print(f"{name} Recall: {recall}")
        print(f"{name} F1: {f1}\n")

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
CURRENT THRESHOLD: 0.01
NN_64_32_1 Accuracy: 0.38943248532289626
NN_64_32_1 Precision: 0.0891812865497076
NN_64_32_1 Recall: 0.9838709677419355
NN_64_32_1 F1: 0.16353887399463807

CURRENT THRESHOLD: 0.028163265306122447
NN_64_32_1 Accuracy: 0.4628180039138943
NN_64_32_1 Precision: 0.09484193011647254
NN_64_32_1 Recall: 0.9193548387096774
NN_64_32_1 F1: 0.17194570135746606

CURRENT THRESHOLD: 0.0463265306122449
NN_64_32_1 Accuracy: 0.5117416829745597
NN_64_32_1 Precision: 0.10200364298724955
NN_64_32_1 Recall: 0.9032258064516129
NN_64_32_1 F1: 0.18330605564648117

CURRENT THRESHOLD: 0.06448979591836734
NN_64_32_1 Accuracy: 0.5371819960861057
NN_64_32_1 Precision: 0.10556621880998081
NN_64_32_1 Recall: 0.8870967741935484
NN_64_32_1 F1: 0.18867924528301888

CURRENT THRESHOLD: 0.08265306122448979
NN_64_32_1 Accuracy: 0.5606653620352251
NN_64_32_1 Precision: 0.1075050709939148
NN_64_32_1 Recall: 0.8548387096774194
NN_64_32_1 F1: 0.19099099099099098

CU

c:\Users\Dell\anaconda\envs\tensorflow\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\anaconda\envs\tensorflow\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Dell\anaconda\envs\tensorflow\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
CURRENT THRESHOLD: 0.01
NN_64_64_1 Accuracy: 0.3385518590998043
NN_64_64_1 Precision: 0.08401084010840108
NN_64_64_1 Recall: 1.0
NN_64_64_1 F1: 0.155

CURRENT THRESHOLD: 0.028163265306122447
NN_64_64_1 Accuracy: 0.3933463796477495
NN_64_64_1 Precision: 0.08849557522123894
NN_64_64_1 Recall: 0.967741935483871
NN_64_64_1 F1: 0.16216216216216217

CURRENT THRESHOLD: 0.0463265306122449
NN_64_64_1 Accuracy: 0.4246575342465753
NN_64_64_1 Precision: 0.09034267912772585
NN_64_64_1 Recall: 0.9354838709677419
NN_64_64_1 F1: 0.16477272727272727

CURRENT THRESHOLD: 0.06448979591836734
NN_64_64_1 Accuracy: 0.45792563600782776
NN_64_64_1 Precision: 0.09271523178807947
NN_64_64_1 Recall: 0.9032258064516129
NN_64_64_1 F1: 0.16816816816816818

CURRENT THRESHOLD: 0.08265306122448979
NN_64_64_1 Accuracy: 0.49510763209393344
NN_64_64_1 Precision: 0.0989399293286219
NN_64_64_1 Recall: 0.9032258064516129
NN_64_64_1 F1: 0.17834394904458598

CURRENT THRESHOLD: 0.100816326